# 03 · Validación de códigos CUPS

**Proyecto:** LINE — Auditor Médico Digital · Health & Life IPS SAS

Valida los códigos CUPS de forma automatizada:
1. Códigos facturados que no aparecen en historia clínica (y viceversa).
2. Códigos nulos o mal formateados (formato interno: 6 dígitos).
3. Códigos fuera del catálogo interno de la empresa.
4. A nivel de fila: ítems facturados sin el mismo CUPS en la HC de su atención.

| Entrada | Salida |
|---|---|
| `data/processed/hc_detalle_clean.csv`, `prefactura_clean.csv` | `outputs/tables/validacion_cups.csv` |
| | `outputs/tables/catalogo_cups_interno.csv` |
| | `outputs/tables/prefactura_items_sin_mismo_cups_en_hc.csv` |

**Nota:** la empresa no entregó catálogo CUPS oficial. El catálogo interno se
define como la **unión** de códigos usados en HC y prefactura, y queda
exportado para que Health & Life lo confirme o reemplace.

## 0 · Configuración (celda autocontenida)

In [5]:
from pathlib import Path
import os
import re
import time

import pandas as pd

try:
    ROOT = Path(__file__).resolve().parents[1]
except NameError:
    ROOT = Path.cwd()
    if ROOT.name in ("notebooks", "src"):
        ROOT = ROOT.parent

DATA_PROC = ROOT / "data" / "processed"
OUT_TAB = ROOT / "outputs" / "tables"
OUT_TAB.mkdir(parents=True, exist_ok=True)

try:
    display  # noqa: B018
except NameError:
    display = print


def reintentar(fn, intentos=6, espera=0.5):
    for _i in range(intentos):
        try:
            return fn()
        except OSError:
            if _i == intentos - 1:
                raise
            time.sleep(espera * (2 ** _i))


def to_csv_seguro(df, path, **kw):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    reintentar(lambda: df.to_csv(tmp, **kw))
    reintentar(lambda: os.replace(tmp, path))


ENTRADAS = [DATA_PROC / "hc_detalle_clean.csv", DATA_PROC / "prefactura_clean.csv"]
_faltan = [str(p_) for p_ in ENTRADAS if not p_.exists()]
assert not _faltan, (
    "FALTAN INSUMOS:\n  - " + "\n  - ".join(_faltan)
    + "\n→ Ejecuta primero el notebook 01_limpieza.ipynb."
)

hc = pd.read_csv(DATA_PROC / "hc_detalle_clean.csv", dtype=str)
pf = pd.read_csv(DATA_PROC / "prefactura_clean.csv", dtype=str)
print(f"HC: {len(hc):,} filas | Prefactura: {len(pf):,} filas")

HC: 3,058 filas | Prefactura: 2,974 filas


## 1 · Catálogo interno y validación código a código
Estados posibles:
- **OK** — presente en HC y prefactura, formato válido.
- **SOLO_HC** — se registra pero nunca se factura (¿servicio no facturable o fuga sistemática?).
- **SOLO_PREFACTURA** — se factura sin aparecer en HC (riesgo de glosa).
- **NO_ESTANDAR_EN_USO** — formato inválido pero uso consistente en ambas fuentes
  (variante interna; requiere revisión con la empresa, no se descarta).
- **MAL_FORMATEADO** — formato inválido y una sola fuente (descartable si la
  empresa no lo reconoce).

In [6]:
FORMATO = re.compile(r"^\d{6}$")


def formato_ok(code):
    return bool(FORMATO.fullmatch(str(code))) if pd.notna(code) else False


cups_hc = set(hc["codigo_cups"].dropna())
cups_pf = set(pf["codigo_cups_facturado"].dropna())
catalogo = sorted(cups_hc | cups_pf)

desc_hc = hc.dropna(subset=["codigo_cups"]).groupby("codigo_cups")["descripcion"].first()
desc_pf = pf.dropna(subset=["codigo_cups_facturado"]).groupby("codigo_cups_facturado")[
    "descripcion_servicio_facturado"].first()

filas = []
for code in catalogo:
    en_hc, en_pf, fmt = code in cups_hc, code in cups_pf, formato_ok(code)
    usos_hc = int((hc["codigo_cups"] == code).sum())
    usos_pf = int((pf["codigo_cups_facturado"] == code).sum())
    if not fmt and en_hc and en_pf:
        estado = "NO_ESTANDAR_EN_USO"
        justif = ("No cumple el formato CUPS de 6 dígitos PERO se usa consistentemente en HC y "
                  "prefactura: variante interna. REQUIERE_REVISION con la empresa; filas conservadas.")
    elif not fmt:
        estado = "MAL_FORMATEADO"
        justif = "Formato inválido y una sola fuente. DESCARTABLE si la empresa no lo reconoce."
    elif en_hc and en_pf:
        estado, justif = "OK", "Presente en HC y prefactura; código activo del catálogo interno."
    elif en_hc:
        estado = "SOLO_HC"
        justif = "Se registra clínicamente pero nunca se factura; revisar si es fuga sistemática."
    else:
        estado = "SOLO_PREFACTURA"
        justif = "Se factura sin aparecer en HC; riesgo de glosa. REQUIERE_REVISION."
    filas.append({"codigo_cups": code, "descripcion": desc_hc.get(code, desc_pf.get(code, "")),
                  "en_historia_clinica": en_hc, "en_prefactura": en_pf, "formato_valido": fmt,
                  "usos_hc": usos_hc, "usos_prefactura": usos_pf,
                  "estado": estado, "justificacion": justif})

catalogo_df = pd.DataFrame(filas)
display(catalogo_df[["codigo_cups", "descripcion", "usos_hc", "usos_prefactura", "estado"]])
print(catalogo_df["estado"].value_counts().to_string())

,codigo_cups,descripcion,usos_hc,usos_prefactura,estado
0,391013,Facoemulsificacion con lente intraocular,94,79,OK
1,391121,Apendicectomia,65,68,OK
2,391201,Angioplastia coronaria,63,63,OK
3,739001,Atencion del parto,46,57,OK
4,871111,Radiografia de torax,170,149,OK
5,881201,Ecografia abdominal total,125,124,OK
6,881332,Ecografia obstetrica,111,112,OK
7,890201,Consulta de primera vez medicina general,474,455,OK
8,890201-M,Terapia de rehidratacion oral supervisada,52,58,NO_ESTANDAR_EN_USO
9,890205,Consulta medica general,1,0,SOLO_HC


estado
OK                    17
SOLO_HC                2
NO_ESTANDAR_EN_USO     1


## 2 · Validación fila a fila por atención
Ítems facturados cuyo CUPS no aparece en la HC de **su misma atención** — la
materia prima de las alertas CODIGO_NO_COINCIDE y SIN_SOPORTE_CLINICO.

In [7]:
pf_hc = pf.merge(hc[["id_atencion", "codigo_cups"]].drop_duplicates(),
                 left_on=["id_atencion", "codigo_cups_facturado"],
                 right_on=["id_atencion", "codigo_cups"],
                 how="left", indicator=True)
sin_soporte = pf_hc[pf_hc["_merge"] == "left_only"]
print(f"Ítems facturados sin el mismo CUPS en la HC de su atención: {len(sin_soporte):,} de {len(pf):,}")

Ítems facturados sin el mismo CUPS en la HC de su atención: 169 de 2,974


## 3 · Exportar tablas de validación → `outputs/tables/`

In [8]:
to_csv_seguro(catalogo_df, OUT_TAB / "validacion_cups.csv", index=False)
to_csv_seguro(catalogo_df[catalogo_df["estado"] == "OK"][["codigo_cups", "descripcion"]],
              OUT_TAB / "catalogo_cups_interno.csv", index=False)
to_csv_seguro(sin_soporte[["id_prefactura", "id_atencion", "codigo_cups_facturado",
                           "descripcion_servicio_facturado", "cantidad_facturada", "valor_total"]],
              OUT_TAB / "prefactura_items_sin_mismo_cups_en_hc.csv", index=False)
print(f"3 tablas exportadas a {OUT_TAB}")

3 tablas exportadas a c:\Users\MIGI\Desktop\LINE\outputs\tables


## ✅ Verificación de cierre
Deben existir las 3 tablas en `outputs/tables/`. Hallazgo esperado con los
datos actuales: 17 códigos OK + 1 `NO_ESTANDAR_EN_USO` (`890201-M`, que la
empresa usa para "Terapia de rehidratación oral supervisada" — un código
inventado que cualquier EPS glosaría).